# 03 - Speed Safety Score, Risk Tiers, and Sensitivity Analysis

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import geopandas as gpd

from src.score import BASE_WEIGHTS, score_and_classify, run_sensitivity_analysis, INSUFFICIENT_DATA_TIER

pd.set_option('display.max_columns', None)

reliable = gpd.read_parquet('../data/processed/_reliable_features.parquet')
reliable = gpd.GeoDataFrame(reliable, geometry='geometry', crs='EPSG:4326')
low_confidence = gpd.read_parquet('../data/processed/_low_confidence.parquet')
low_confidence = gpd.GeoDataFrame(low_confidence, geometry='geometry', crs='EPSG:4326')

## Score and classify

Weights: `speed_gap_norm` 0.25, `road_mismatch` 0.25, `bio_risk` 0.35, and `population_exposure` 0.15. The anchored speed and mismatch components remain comparable across countries; `vru_exposure` scales `bio_risk` rather than appearing separately, and population contributes a visible 0-15 points. `confidence_weight` remains a reporting flag and does not discount risk. Tiers: High >= 70, Medium >= 40, Low < 40; low-confidence segments are labelled 'Insufficient data'.

In [2]:
reliable = score_and_classify(reliable, BASE_WEIGHTS)
print('risk_tier value counts (reliable segments):')
print(reliable['risk_tier'].value_counts())
print(f"\nlow_confidence segments (all '{INSUFFICIENT_DATA_TIER}'): {len(low_confidence)}")

risk_tier value counts (reliable segments):
risk_tier
Low risk       8991
Medium risk    5476
High risk        79
Name: count, dtype: int64

low_confidence segments (all 'Insufficient data'): 55420


In [3]:
cols = ['segment_id', 'road_class', 'SpeedLimit', 'F85thPercentileSpeed', 'speed_gap',
        'risk_tier', 'speed_safety_score', 'RoadLength_km', 'mapillary_url']
display(reliable.nlargest(10, 'speed_safety_score')[cols])

,segment_id,road_class,SpeedLimit,F85thPercentileSpeed,speed_gap,risk_tier,speed_safety_score,RoadLength_km,mapillary_url
9121,47941,secondary,90.0,106.666667,16.666667,High risk,77.9,4.481631,https://www.mapillary.com/app/?lat=15.32532934...
4409,31570,secondary,90.0,106.000000,16.000000,High risk,77.2,2.315821,https://www.mapillary.com/app/?lat=18.24769807...
7057,39416,secondary,90.0,106.000000,16.000000,High risk,77.2,0.230643,https://www.mapillary.com/app/?lat=18.32868375...
10238,52083,secondary,90.0,103.500000,13.500000,High risk,77.2,0.719354,https://www.mapillary.com/app/?lat=6.398783630...
8516,44747,secondary,90.0,106.000000,16.000000,High risk,76.7,0.368242,https://www.mapillary.com/app/?lat=17.6786756&...
3721,28020,primary,90.0,110.000000,20.000000,High risk,76.4,13.065160,https://www.mapillary.com/app/?lat=13.12949611...
3724,28035,primary,90.0,110.000000,20.000000,High risk,76.4,1.713936,https://www.mapillary.com/app/?lat=12.6915324&...
7791,41752,secondary,90.0,104.000000,14.000000,High risk,74.7,1.404344,https://www.mapillary.com/app/?lat=16.99589104...
8720,45568,secondary,90.0,102.000000,12.000000,High risk,74.3,0.621569,https://www.mapillary.com/app/?lat=14.88283442...
7043,39349,secondary,90.0,104.000000,14.000000,High risk,74.2,0.761706,https://www.mapillary.com/app/?lat=18.32813610...


## Validate against `RankedPercentile`

`RankedPercentile` ranks segments by **travel volume share**, not by safety risk — a busy, well-managed motorway can rank high on traffic but low on risk. A weak or negative correlation is therefore informative about what each metric measures, not necessarily a bug in the score.

In [4]:
valid_corr = reliable[['speed_safety_score', 'RankedPercentile']].dropna()
corr = valid_corr['speed_safety_score'].corr(valid_corr['RankedPercentile'])
print(f'Pearson correlation speed_safety_score vs RankedPercentile: {corr:.4f}')
if corr < 0.3:
    print('WARNING: correlation with RankedPercentile is below 0.3 -- see methodology.md for discussion.')
else:
    print('Correlation is positive and above the 0.3 sanity threshold.')

Pearson correlation speed_safety_score vs RankedPercentile: -0.0138


## Sensitivity analysis

Each weight is varied +-0.10 in 0.05 steps, keeping all three weights summing to 1.0. For every valid combination, the top-20%-by-score segments are compared back to the baseline top 20%.

In [5]:
sens = run_sensitivity_analysis(reliable, BASE_WEIGHTS)
print(f"{sens['n_variants']} valid weight variants tested.")
print(f"Average top-20% overlap with baseline: {sens['average_overlap_pct']:.2f}%")
if sens['robust']:
    print('Score is robust to weight variation')
else:
    print('WARNING: score is sensitive to weight variation')
    print('Segments that flip in/out of the top 20% most often:')
    flippers = reliable.loc[sens['flip_counts'].index[:15], ['segment_id', 'country', 'speed_safety_score']]
    flippers['flip_count'] = sens['flip_counts'].values[:15]
    display(flippers)

18 valid weight variants tested.
Average top-20% overlap with baseline: 79.65%
Score is robust to weight variation


In [6]:
reliable.to_parquet('../data/processed/_reliable_scored.parquet')
low_confidence.to_parquet('../data/processed/_low_confidence_final.parquet')
import json
with open('../data/processed/_sensitivity_summary.json', 'w') as f:
    json.dump({'average_overlap_pct': sens['average_overlap_pct'], 'n_variants': sens['n_variants'],
               'correlation_with_ranked_percentile': corr}, f)
print('Saved intermediate parquet/json for notebook 04.')

Saved intermediate parquet/json for notebook 04.
